In [3]:
import numpy as np
import pandas as pd
import os
import joblib
import dotenv

In [4]:
dotenv.load_dotenv()

DATASETS_PATH = os.getenv("DATASETS_PATH")
MODELS_DIR = os.getenv("MODELS_DIR")
OUTPUT_DIR = os.path.join(os.getenv("REPORTS_DIR"), "predictions")

# Create output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

from helpers import load_dataset, load_model, load_test_dataset, load_merged_test_datasets

In [5]:
def predict_for_modality_and_split(modality, split, model, X, y):
    print(f"Predicting for modality: {modality}, split: {split}")

    # Get predictions
    y_pred_proba = model.predict_proba(X)
    y_pred = model.predict(X)

    pred_df = pd.DataFrame(y_pred, columns=["predicted_label"], index=X.index)
    proba_df = pd.DataFrame(y_pred_proba, columns=model.classes_, index=X.index)

    # Make first column the true label, then predicted label, then probabilities
    proba_df.insert(0, "true_label", y)
    proba_df.insert(1, "predicted_label", y_pred)

    # Save predicted probabilities to CSV
    output_path = os.path.join(OUTPUT_DIR, f"{modality}_{split}_predictions.csv")
    proba_df.to_csv(output_path)
    print(f"Predicted probabilities saved to: {output_path}")

    return y_pred, y_pred_proba

def predict_for_modality(modality):
    model = load_model(modality)

    X_test, y_test = load_test_dataset(modality, "test")
    X_cal, y_cal = load_test_dataset(modality, "calibrate")
    X_merged, y_merged = load_merged_test_datasets(modality)

    predict_for_modality_and_split(modality, "test", model, X_test, y_test)
    predict_for_modality_and_split(modality, "calibrate", model, X_cal, y_cal)
    X_merged, y_merged = predict_for_modality_and_split(modality, "merged_test_calibrate", model, X_merged, y_merged)
    return X_merged, y_merged


X_gex, y_gex = predict_for_modality("gex")
X_dnam, y_dnam = predict_for_modality("dnam")

Predicting for modality: gex, split: test
Predicted probabilities saved to: /home/mariya/Data/pallas/reports/predictions/gex_test_predictions.csv
Predicting for modality: gex, split: calibrate
Predicted probabilities saved to: /home/mariya/Data/pallas/reports/predictions/gex_calibrate_predictions.csv
Predicting for modality: gex, split: merged_test_calibrate
Predicted probabilities saved to: /home/mariya/Data/pallas/reports/predictions/gex_merged_test_calibrate_predictions.csv
Predicting for modality: dnam, split: test
Predicted probabilities saved to: /home/mariya/Data/pallas/reports/predictions/dnam_test_predictions.csv
Predicting for modality: dnam, split: calibrate
Predicted probabilities saved to: /home/mariya/Data/pallas/reports/predictions/dnam_calibrate_predictions.csv
Predicting for modality: dnam, split: merged_test_calibrate
Predicted probabilities saved to: /home/mariya/Data/pallas/reports/predictions/dnam_merged_test_calibrate_predictions.csv
